In [ ]:
import os

SEED = 42

os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

import random
import numpy as np
import tensorflow as tf

random.seed(SEED)
np.random.seed(SEED)

tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

print("Deterministic seed =", SEED)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if 'Raw measurements' in f:
            print("✅", os.path.join(root, f))

In [ ]:
import pandas as pd

PATH = "/content/drive/MyDrive/Raw measurements. csv"

df = pd.read_csv(PATH)

print("Dataset loaded ✅")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
print(sorted(df['classtype'].dropna().unique()))

In [ ]:
# ==============================
# STEP 1: Create Timestamp
# ==============================

import pandas as pd
import numpy as np

df['Timestamp'] = pd.to_datetime(
    df['Date'].astype(str) + ' ' + df['Time'].astype(str),
    errors='coerce'
)

# Drop rows without a valid timestamp
df = df.dropna(subset=['Timestamp']).copy()

# Sort each device chronologically within each room
df = df.sort_values(
    ['roomcode', 'devcode', 'Timestamp']
).reset_index(drop=True)

print("✅ Timestamp created")
print("Rows:", len(df))
print("Rooms:", df['roomcode'].nunique())
print("Devices:", df['devcode'].nunique())

print("\nFirst rows:")
print(df[['roomcode', 'devcode', 'Timestamp']].head(10))

In [ ]:
# ==========================================
# STEP 2: Check sampling intervals per device
# ==========================================

temp = df[['roomcode', 'devcode', 'Timestamp']].copy()

temp['gap_min'] = (
    temp.groupby(['roomcode', 'devcode'])['Timestamp']
        .diff()
        .dt.total_seconds() / 60
)

# Exclude implausible values and long gaps from the distribution plot
gaps = temp.loc[
    (temp['gap_min'] > 0) &
    (temp['gap_min'] <= 60),
    'gap_min'
]

print("Median interval:", round(gaps.median(), 2), "minutes")

print("\nMost common intervals:")
print(
    gaps.round(2)
        .value_counts()
        .head(20)
)

In [ ]:
# ============================================================
# STEP 3 FINAL: CONTINUOUS 3-MIN RESAMPLING
# No device mixing + No interpolation across gaps > 15 minutes
# ============================================================

import pandas as pd
import numpy as np

SENSOR_COLS = ['CO2', 'PM2.5', 'PM10', 'T', 'RH']
MAX_GAP_MIN = 15
RESAMPLE_RULE = '3min'

# Work from the original data
work = df[
    ['roomcode', 'devcode', 'Timestamp'] + SENSOR_COLS
].copy()

work = work.dropna(subset=['roomcode', 'devcode', 'Timestamp'])
work = work.sort_values(
    ['roomcode', 'devcode', 'Timestamp']
).reset_index(drop=True)

# ------------------------------------------------------------
# 1. Identify interruptions within each device
# ------------------------------------------------------------
gap = (
    work.groupby(['roomcode', 'devcode'])['Timestamp']
        .diff()
        .dt.total_seconds()
        .div(60)
)

new_segment = (
    gap.isna() |
    (gap > MAX_GAP_MIN)
)

work['segment_id'] = (
    new_segment.groupby(
        [work['roomcode'], work['devcode']]
    ).cumsum()
)

print("Original rows :", len(work))
print("Segments      :", work[
    ['roomcode','devcode','segment_id']
].drop_duplicates().shape[0])

# ------------------------------------------------------------
# 2. Resampling within each continuous segment only
# ------------------------------------------------------------
parts = []

for (room, dev, seg), g in work.groupby(
    ['roomcode', 'devcode', 'segment_id'],
    sort=False
):

    if len(g) < 2:
        continue

    g = (
        g.set_index('Timestamp')[SENSOR_COLS]
        .resample(RESAMPLE_RULE)
        .mean()
    )

    # Short gaps only: at most 4 missing points
    MAX_MISSING_BINS = 4

    for col in SENSOR_COLS:

        s = g[col].copy()

        interp = s.interpolate(
            method='linear',
            limit_area='inside'
        )

        is_nan = s.isna()

        run_id = (is_nan != is_nan.shift()).cumsum()

        run_sizes = (
            is_nan.groupby(run_id)
            .transform('sum')
        )

        short_gap = is_nan & (run_sizes <= MAX_MISSING_BINS)

        s.loc[short_gap] = interp.loc[short_gap]

        g[col] = s

    # Drop rows that still have a missing sensor value
    g = g.dropna(subset=SENSOR_COLS)

    if len(g) == 0:
        continue

    # Restore Timestamp from the index
    g = g.rename_axis('Timestamp').reset_index()

    g['roomcode'] = room
    g['devcode'] = dev
    g['segment_id'] = seg

    parts.append(g)

# ------------------------------------------------------------
# 3. Final resampled dataset
# ------------------------------------------------------------
df_resampled = pd.concat(parts, ignore_index=True)

df_resampled = df_resampled[
    ['roomcode', 'devcode', 'segment_id',
     'Timestamp'] + SENSOR_COLS
]

df_resampled = df_resampled.sort_values(
    ['roomcode', 'devcode', 'segment_id', 'Timestamp']
).reset_index(drop=True)
# Re-segment after removing long missing runs
df_resampled = df_resampled.sort_values(
    ['roomcode', 'devcode', 'segment_id', 'Timestamp']
).reset_index(drop=True)

gap_after = (
    df_resampled
    .groupby(['roomcode', 'devcode'])['Timestamp']
    .diff()
    .dt.total_seconds()
    .div(60)
)

new_break = gap_after.isna() | (gap_after > 15)

df_resampled['segment_id'] = (
    new_break
    .groupby([
        df_resampled['roomcode'],
        df_resampled['devcode']
    ])
    .cumsum()
    .astype('int32')
)
# ------------------------------------------------------------
# 4. Verification
# ------------------------------------------------------------
check_gap = (
    df_resampled
    .groupby(['roomcode','devcode','segment_id'])['Timestamp']
    .diff()
    .dt.total_seconds()
    .div(60)
    .dropna()
)

print("\n✅ Corrected resampling completed")
print("Resampled rows :", len(df_resampled))
print("Rooms          :", df_resampled['roomcode'].nunique())
print("Devices        :", df_resampled['devcode'].nunique())

print("\nIntervals inside segments:")
print(check_gap.value_counts().head())

print("\nMissing values:")
print(df_resampled[SENSOR_COLS].isna().sum())

In [ ]:
# ============================================================
# CLEANING — AFTER 3-MIN RESAMPLING, BEFORE STEP 4
# ============================================================

import numpy as np
import pandas as pd

clean = df_resampled.copy()

print("Rows before cleaning:", len(clean))

# ------------------------------------------------------------
# 1) Remove duplicated rows
# ------------------------------------------------------------
before = len(clean)
clean = clean.drop_duplicates()
print("Duplicates removed:", before - len(clean))

# ------------------------------------------------------------
# 2) Replace inf / -inf with NaN
# ------------------------------------------------------------
features = ["CO2", "PM2.5", "PM10", "T", "RH"]

clean[features] = clean[features].replace(
    [np.inf, -np.inf],
    np.nan
)

# ------------------------------------------------------------
# 3) Remove physically impossible readings
# ------------------------------------------------------------

# Concentrations cannot be negative
clean.loc[clean["CO2"] < 0, "CO2"] = np.nan
clean.loc[clean["PM2.5"] < 0, "PM2.5"] = np.nan
clean.loc[clean["PM10"] < 0, "PM10"] = np.nan

# Relative Humidity must be between 0 and 100%
clean.loc[
    (clean["RH"] < 0) | (clean["RH"] > 100),
    "RH"
] = np.nan

# ------------------------------------------------------------
# 4) Count missing/invalid values after cleaning
# ------------------------------------------------------------
print("\nInvalid / missing values detected:")
print(clean[features].isna().sum())

# ------------------------------------------------------------
# 5) Interpolate ONLY inside the same room/device/segment
# ------------------------------------------------------------
group_cols = ["roomcode", "devcode", "segment_id"]

clean = clean.sort_values(
    group_cols + ["Timestamp"]
).reset_index(drop=True)

clean[features] = (
    clean
    .groupby(group_cols, group_keys=False)[features]
    .apply(
        lambda g: g.interpolate(
            method="linear",
            limit_direction="both"
        )
    )
)

# ------------------------------------------------------------
# 6) Drop any rows still missing required variables
# ------------------------------------------------------------
before = len(clean)

clean = clean.dropna(
    subset=features
).reset_index(drop=True)

print("\nRows removed because values could not be recovered:",
      before - len(clean))

# ------------------------------------------------------------
# 7) Final checks
# ------------------------------------------------------------
print("\nFinal missing values:")
print(clean[features].isna().sum())

print("\nFinal ranges:")
print(clean[features].agg(["min", "max"]))

print("\nRows after cleaning:", len(clean))

# Replace df_resampled with cleaned version
df_resampled = clean.copy()

print("\nCLEANING COMPLETE ✅")

In [ ]:
# ============================================================
# STATISTICAL ANALYSIS + OUTLIER DETECTION
# AFTER CLEANING, BEFORE STEP 4
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

features = ["CO2", "PM2.5", "PM10", "T", "RH"]

# ------------------------------------------------------------
# 1) Descriptive Statistics
# ------------------------------------------------------------
stats = df_resampled[features].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

stats["median"] = df_resampled[features].median()
stats["variance"] = df_resampled[features].var()
stats["skewness"] = df_resampled[features].skew()

print("DESCRIPTIVE STATISTICS")
display(stats)

# ------------------------------------------------------------
# 2) IQR Outlier Detection — COUNT ONLY, DO NOT DELETE
# ------------------------------------------------------------
print("\nIQR OUTLIER DETECTION")
print("NOTE: These values are detected only, NOT removed.\n")

outlier_summary = []

for col in features:
    Q1 = df_resampled[col].quantile(0.25)
    Q3 = df_resampled[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    mask = (df_resampled[col] < lower) | (df_resampled[col] > upper)

    count = int(mask.sum())
    percent = 100 * count / len(df_resampled)

    outlier_summary.append({
        "Feature": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower_Bound": lower,
        "Upper_Bound": upper,
        "Outlier_Count": count,
        "Outlier_Percent": percent
    })

outlier_df = pd.DataFrame(outlier_summary)

display(outlier_df)

# ------------------------------------------------------------
# 3) Correlation Matrix
# ------------------------------------------------------------
print("\nPEARSON CORRELATION MATRIX")
corr = df_resampled[features].corr(method="pearson")
display(corr)

# ------------------------------------------------------------
# 4) Histograms
# ------------------------------------------------------------
for col in features:
    plt.figure(figsize=(7, 4))
    plt.hist(df_resampled[col].dropna(), bins=60)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

# ------------------------------------------------------------
# 5) Boxplots
# ------------------------------------------------------------
for col in features:
    plt.figure(figsize=(7, 2.5))
    plt.boxplot(df_resampled[col].dropna(), vert=False)
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)
    plt.show()

print("\nSTATISTICAL ANALYSIS COMPLETE ✅")
print("IMPORTANT: IQR outliers were detected only — nothing was deleted.")

In [ ]:
# ============================================================
# STEP 4: 8-HOUR PREDICTED MOVING AVERAGES
# Within each room + device + continuous segment only
# ============================================================

import pandas as pd
import numpy as np

data = df_resampled.copy()

GROUP = ['roomcode', 'devcode', 'segment_id']

data = data.sort_values(
    GROUP + ['Timestamp']
).reset_index(drop=True)

parts_step4 = []

for _, g in data.groupby(GROUP, sort=False):

    g = g.sort_values('Timestamp').copy()
    g = g.set_index('Timestamp')

    for col in ['CO2', 'PM2.5', 'PM10']:

        # Past 4-hour average
        avg4 = g[col].rolling(
            '4h',
            min_periods=2
        ).mean()

        # Past 1-hour average
        avg1 = g[col].rolling(
            '1h',
            min_periods=2
        ).mean()

        # IAQI-S predicted 8-hour average
        g[f'{col}_8h'] = (4 * avg4 + 4 * avg1) / 8

    g = g.reset_index()
    parts_step4.append(g)

data = pd.concat(parts_step4, ignore_index=True)

# Drop rows without complete averages
before = len(data)

data = data.dropna(
    subset=['CO2_8h', 'PM2.5_8h', 'PM10_8h']
).reset_index(drop=True)

print("✅ STEP 4 completed")
print("Before :", before)
print("After  :", len(data))

print(
    data[
        ['roomcode', 'devcode', 'segment_id', 'Timestamp',
         'CO2_8h', 'PM2.5_8h', 'PM10_8h']
    ].head()
)

In [ ]:
# ============================================================
# STEP 5 FINAL: IAQI-S Labels
# Continuous linear interpolation — no gaps, no rounding
# ============================================================

import numpy as np
import pandas as pd

N_BEFORE = len(data)
print("Rows before IAQI-S:", N_BEFORE)


# ------------------------------------------------------------
# 1. Continuous IAQI interpolation
# ------------------------------------------------------------

def iaqi_continuous(x, concentration_points):
    """
    Continuous linear interpolation from pollutant concentration
    to IAQI-S index.
    """
    x = np.asarray(x, dtype=np.float32)

    iaqi_points = np.array(
        [0, 50, 100, 200, 300, 400, 500],
        dtype=np.float32
    )

    concentration_points = np.asarray(
        concentration_points,
        dtype=np.float32
    )

    out = np.interp(
        x,
        concentration_points,
        iaqi_points
    )

    return out.astype(np.float32)


# ------------------------------------------------------------
# 2. IAQI-S concentration breakpoints
# ------------------------------------------------------------

PM10_POINTS = [
    0, 20, 75, 150, 220, 300, 380
]

PM25_POINTS = [
    0, 10, 35, 70, 100, 140, 180
]

CO2_POINTS = [
    300, 700, 1000, 2000, 3000, 4000, 5000
]


# ------------------------------------------------------------
# 3. Individual IAQI
# ------------------------------------------------------------

data["IAQI_PM10"] = iaqi_continuous(
    data["PM10_8h"].to_numpy(),
    PM10_POINTS
)

data["IAQI_PM25"] = iaqi_continuous(
    data["PM2.5_8h"].to_numpy(),
    PM25_POINTS
)

data["IAQI_CO2"] = iaqi_continuous(
    data["CO2_8h"].to_numpy(),
    CO2_POINTS
)


# ------------------------------------------------------------
# 4. Composite IAQI-S
# ------------------------------------------------------------

idx = data[
    ["IAQI_PM10", "IAQI_PM25", "IAQI_CO2"]
].to_numpy(dtype=np.float32)

# Maximum individual IAQI
base = np.max(idx, axis=1)

# Number of pollutants above Caution threshold
n_caution = np.sum(idx > 100, axis=1)

# EP1
ep1 = np.zeros(len(data), dtype=np.float32)
ep1[n_caution == 2] = 50
ep1[n_caution == 3] = 75

# EP2
ep2 = (
    np.sum((idx > 100) & (idx <= 200), axis=1) * 5
    + np.sum((idx > 200) & (idx <= 300), axis=1) * 10
    + np.sum((idx > 300) & (idx <= 400), axis=1) * 20
    + np.sum(idx > 400, axis=1) * 40
)

# Final composite
data["IAQI_Composite"] = np.clip(
    base + ep1 + ep2,
    0,
    500
)


# ------------------------------------------------------------
# 5. Six IAQI-S classes
# ------------------------------------------------------------

labels = np.array([
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
])

score = data["IAQI_Composite"].to_numpy()

# <=50, >50-100, >100-200, >200-300, >300-400, >400
class_id = np.digitize(
    score,
    bins=[50, 100, 200, 300, 400],
    right=True
)

data["IAQI_Class"] = labels[class_id]


# ------------------------------------------------------------
# 6. Final verification
# ------------------------------------------------------------

N_AFTER = len(data)

print("\n✅ IAQI-S Labels created")
print("Rows before:", N_BEFORE)
print("Rows after :", N_AFTER)

assert N_AFTER == N_BEFORE, "ERROR: number of rows changed!"


print("\nClass distribution:")

counts = data["IAQI_Class"].value_counts()

for cls in labels:
    n = counts.get(cls, 0)
    p = n / N_AFTER * 100

    print(f"{cls:16s}: {n:,} ({p:.2f}%)")

In [ ]:
# ============================================================
# STEP 6 FINAL:
# Fixed + deterministic stratified ROOM-BASED split
# Uses only labels that will actually survive WINDOW=10
# No room overlap
# ============================================================

import numpy as np
import pandas as pd

SEED = 42
WINDOW = 10
N_VAL = 5
N_TEST = 5

LABELS = [
    'Good',
    'Moderate',
    'Caution',
    'Unhealthy',
    'Very Unhealthy',
    'Hazardous'
]

GROUP = ['roomcode', 'devcode', 'segment_id']

# Physical room = room code without the retrofit-period suffix (-a/-b/-c)
data['room_group'] = (
    data['roomcode'].astype(str).str.replace(r'[abc]$', '', regex=True)
)

# ------------------------------------------------------------
# 1. Find labels that are actually eligible after WINDOW=10
# ------------------------------------------------------------
eligible_idx = []

for _, g in data.groupby(GROUP, sort=False):

    g = g.sort_values('Timestamp')

    if len(g) < WINDOW:
        continue

    eligible_idx.extend(
        g.index[WINDOW - 1:].tolist()
    )

eligible = data.loc[
    eligible_idx,
    ['room_group', 'IAQI_Class']
].copy()

all_rooms = sorted(
    data['room_group'].dropna().unique().tolist(),
    key=lambda x: str(x)
)

room_counts = (
    eligible
    .groupby(['room_group', 'IAQI_Class'])
    .size()
    .unstack(fill_value=0)
    .reindex(index=all_rooms, fill_value=0)
    .reindex(columns=LABELS, fill_value=0)
)

# ------------------------------------------------------------
# 2. Scientific check:
# each class must exist in at least 3 different rooms
# if we want Train + Validation + Test all to contain it
# ------------------------------------------------------------
room_support = (room_counts > 0).sum(axis=0)

print("Rooms containing each eligible class:")
print(room_support)

if (room_support < 3).any():
    bad = room_support[room_support < 3]

    raise ValueError(
        "\nCannot create a leakage-free room-based "
        "Train/Validation/Test split containing every class.\n"
        f"Classes with fewer than 3 supporting rooms:\n{bad}"
    )

# ------------------------------------------------------------
# 3. Deterministic room selection
#    Not thousands of random trials
# ------------------------------------------------------------
rng = np.random.default_rng(SEED)

random_order = rng.permutation(all_rooms)

tie_rank = {
    room: i
    for i, room in enumerate(random_order)
}

global_counts = room_counts.sum(axis=0).to_numpy(dtype=float)
global_prop = global_counts / global_counts.sum()

def choose_rooms(candidates, n_pick):

    candidates = list(candidates)

    selected = []
    selected_counts = np.zeros(len(LABELS), dtype=float)

    support = (
        (room_counts.loc[candidates] > 0)
        .sum(axis=0)
        .to_numpy(dtype=float)
    )

    # Start with rarest classes
    class_order = np.argsort(support)

    for j in class_order:

        if selected_counts[j] > 0:
            continue

        available = [
            r for r in candidates
            if r not in selected
            and room_counts.loc[r, LABELS[j]] > 0
        ]

        if not available:
            raise ValueError(
                f"No available room contains class {LABELS[j]}"
            )

        def coverage_score(room):

            row = (
                room_counts
                .loc[room, LABELS]
                .to_numpy(dtype=float)
            )

            present = row > 0

            return np.sum(
                present / (support + 1e-9)
            )

        best_room = max(
            available,
            key=lambda r: (
                coverage_score(r),
                -tie_rank[r]
            )
        )

        selected.append(best_room)

        selected_counts += (
            room_counts
            .loc[best_room, LABELS]
            .to_numpy(dtype=float)
        )

        if len(selected) == n_pick:
            return np.array(selected, dtype=object)

    # Fill remaining rooms while keeping distribution close
    while len(selected) < n_pick:

        best_room = None
        best_score = np.inf

        for room in candidates:

            if room in selected:
                continue
            new_counts = (
                selected_counts
                + room_counts
                  .loc[room, LABELS]
                  .to_numpy(dtype=float)
            )

            if new_counts.sum() == 0:
                score = np.inf
            else:
                new_prop = new_counts / new_counts.sum()

                score = np.mean(
                    np.abs(new_prop - global_prop)
                    / np.sqrt(global_prop + 1e-8)
                )

            if (
                score < best_score
                or (
                    np.isclose(score, best_score)
                    and (
                        best_room is None
                        or tie_rank[room] < tie_rank[best_room]
                    )
                )
            ):
                best_score = score
                best_room = room

        selected.append(best_room)

        selected_counts += (
            room_counts
            .loc[best_room, LABELS]
            .to_numpy(dtype=float)
        )

    return np.array(selected, dtype=object)

# Test first
test_rooms = choose_rooms(
    all_rooms,
    N_TEST
)

remaining = [
    r for r in all_rooms
    if r not in set(test_rooms)
]

# Validation from remaining rooms
val_rooms = choose_rooms(
    remaining,
    N_VAL
)

train_rooms = np.array(
    [
        r for r in remaining
        if r not in set(val_rooms)
    ],
    dtype=object
)

# ------------------------------------------------------------
# 4. Create final data partitions
# ------------------------------------------------------------
train_part = data[
    data['room_group'].isin(train_rooms)
].copy()

val_part = data[
    data['room_group'].isin(val_rooms)
].copy()

test_part = data[
    data['room_group'].isin(test_rooms)
].copy()

# ------------------------------------------------------------
# 5. Final verification using WINDOW-eligible labels
# ------------------------------------------------------------
print("\n✅ FINAL FIXED PHYSICAL-ROOM SPLIT")

print("Train rooms      :", len(train_rooms))
print("Validation rooms :", len(val_rooms))
print("Test rooms       :", len(test_rooms))

print("\nRows:")
print("Train      :", len(train_part))
print("Validation :", len(val_part))
print("Test       :", len(test_part))

print("\nRoom overlap:")
print(
    "Train-Val  :",
    len(set(train_rooms) & set(val_rooms))
)
print(
    "Train-Test :",
    len(set(train_rooms) & set(test_rooms))
)
print(
    "Val-Test   :",
    len(set(val_rooms) & set(test_rooms))
)

for name, rooms_part in [
    ("TRAIN", train_rooms),
    ("VAL", val_rooms),
    ("TEST", test_rooms)
]:

    counts = (
        room_counts
        .loc[list(rooms_part), LABELS]
        .sum(axis=0)
    )

    print(f"\n{name} eligible labels after WINDOW={WINDOW}:")

    for cls in LABELS:
        print(f"{cls:16s}: {int(counts[cls]):,}")

    assert (counts > 0).all(), (
        f"{name} is missing one or more classes"
    )

print("\n✅ All 6 classes exist in Train, Validation and Test")

In [ ]:
print("66 room codes     :", data['roomcode'].nunique())
print("34 physical rooms :", data['room_group'].nunique())

n_tr, n_va, n_te = len(train_part), len(val_part), len(test_part)
print("\nRows  train/val/test :", f"{n_tr:,} / {n_va:,} / {n_te:,}")
print("Sum of three sets   :", f"{n_tr+n_va+n_te:,}")
print("len(data)           :", f"{len(data):,}")
print("Share %             :", [round(x/len(data)*100, 1) for x in (n_tr, n_va, n_te)])

print("\nRooms containing Hazardous (after window):",
      int((room_counts['Hazardous'] > 0).sum()))

In [ ]:
from sklearn.preprocessing import StandardScaler

features = ['CO2', 'PM2.5', 'PM10', 'T', 'RH']

scaler = StandardScaler()

# Fit ONLY on training data
scaler.fit(train_part[features])

# Transform Train / Validation / Test
train_part[features] = scaler.transform(train_part[features])
val_part[features]   = scaler.transform(val_part[features])
test_part[features]  = scaler.transform(test_part[features])

print("✅ Scaling completed")
print("Features:", features)

print("\nTrain mean after scaling:")
print(train_part[features].mean().round(3))

print("\nTrain std after scaling:")
print(train_part[features].std().round(3))

In [ ]:
# ============================================================
# STEP 8 (corrected): Create 10-reading windows
# Same room + same device + same segment ONLY
# Uses train_part / val_part / test_part directly (from STEP 6)
# ============================================================

import numpy as np

WINDOW =10

class_to_id = {
    'Good': 0,
    'Moderate': 1,
    'Caution': 2,
    'Unhealthy': 3,
    'Very Unhealthy': 4,
    'Hazardous': 5
}

def make_windows(df):

    X_all = []
    y_all = []

    # Every sequence must stay within the same room, device and segment
    for _, g in df.groupby(
        ['roomcode', 'devcode', 'segment_id'],
        sort=False
    ):

        g = g.sort_values('Timestamp')

        if len(g) < WINDOW:
            continue

        X = g[features].to_numpy(dtype=np.float32)
        y = g['IAQI_Class'].map(class_to_id).to_numpy()

        windows = np.lib.stride_tricks.sliding_window_view(
            X, WINDOW, axis=0
        ).transpose(0, 2, 1)

        X_all.append(windows)
        y_all.append(y[WINDOW - 1:])   # current-state label

    X_all = np.concatenate(X_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)

    return X_all, y_all


# ⚠️ The correction is here: use train_part / val_part / test_part
# (the same variables produced by the corrected STEP 6), not train_df/val_df/test_df

X_train, y_train = make_windows(train_part)
X_val, y_val     = make_windows(val_part)
X_test, y_test   = make_windows(test_part)

print("✅ 10-reading windows created correctly")

print("\nTrain:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nEach window shape:", X_train.shape[1:])

# ------------------------------------------------------------
# Immediate check: actual class distribution after building the windows
# ------------------------------------------------------------
class_names = ['Good','Moderate','Caution','Unhealthy','Very Unhealthy','Hazardous']

print("\n📊 Actual class distribution after windowing:")
for name, y_set in [("TRAIN", y_train), ("VAL", y_val), ("TEST", y_test)]:
    print(f"\n{name}:")
    for i, cname in enumerate(class_names):
        n = np.sum(y_set == i)
        print(f"  {cname:16s}: {n:,}")

In [ ]:
# ============================================================
# STEP 8b (new): Oversampling of the very rare classes — TRAIN only
# Very Unhealthy and Hazardous only, without touching Val/Test
# ============================================================

import numpy as np

RARE_CLASSES = [4, 5]   # 4=Very Unhealthy, 5=Hazardous
TARGET_COUNT = 5000

rng = np.random.default_rng(42)

X_train_list = [X_train]
y_train_list = [y_train]

for cls in RARE_CLASSES:
    idx = np.where(y_train == cls)[0]
    n_have = len(idx)

    if n_have == 0:
        print(f"⚠️ Warning: class {cls} does not exist in training at all!")
        continue

    n_needed = TARGET_COUNT - n_have
    if n_needed <= 0:
        continue

    chosen = rng.choice(idx, size=n_needed, replace=True)
    X_extra = X_train[chosen].copy()

    noise = rng.normal(0, 0.01, X_extra.shape).astype(np.float32)
    X_extra = X_extra + noise

    y_extra = np.full(n_needed, cls, dtype=y_train.dtype)

    X_train_list.append(X_extra)
    y_train_list.append(y_extra)

    print(f"✅ Class {cls}: was {n_have} → now {n_have + n_needed}")

X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

shuffle_idx = rng.permutation(len(X_train))
X_train = X_train[shuffle_idx]
y_train = y_train[shuffle_idx]

print("\n📊 Final Train distribution after oversampling:")
for i in range(6):
    print(f"  {i}: {np.sum(y_train == i):,}")

print("\n🔒 Check: Val and Test are unchanged")
print("Test Hazardous:", np.sum(y_test == 5))

In [ ]:
# ============================================================
# STEP 9 FINAL
# Oversampling is already used -> NO class weights
# ============================================================

class_weights = None

print("✅ Oversampling only")
print("✅ Class weights disabled")

In [ ]:
# ============================================================
# FINAL MODEL: CNN + LSTM + TCN HYBRID
# (replaces the previous "proposed" model cell)
# NOTE: variable names proposed_model / y_prob_proposed /
# y_pred_proposed are KEPT on purpose so that the downstream
# cells (SHAP, decision layer, dashboard, Firebase, save) run
# without any edit. In the paper this model is called
# "CNN + LSTM + TCN".
# ============================================================
import time
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Activation,
    LSTM,
    Dropout,
    Add,
    GlobalAveragePooling1D,
    Dense
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.metrics import (
    classification_report,
    f1_score
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42
tf.keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# TCN BLOCK
# ------------------------------------------------------------

def tcn_block(x, filters=64, kernel_size=3,
              dilation_rate=2, dropout=0.2):

    residual = x

    x = Conv1D(
        filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        name="TCN_Conv1"
    )(x)

    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Dropout(dropout)(x)

    x = Conv1D(
        filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        name="TCN_Conv2"
    )(x)

    x = BatchNormalization()(x)

    if residual.shape[-1] != filters:
        residual = Conv1D(
            filters,
            kernel_size=1,
            padding="same"
        )(residual)

    x = Add()([x, residual])
    x = Activation("relu")(x)

    return x


# ------------------------------------------------------------
# 1. BUILD MODEL
# Input -> CNN -> LSTM -> TCN -> Dense -> Output
# ------------------------------------------------------------

inputs = Input(
    shape=X_train.shape[1:],
    name="Input"
)

# CNN
x = Conv1D(
    filters=64,
    kernel_size=3,
    padding="same",
    name="CNN_64"
)(inputs)

x = BatchNormalization()(x)
x = Activation("relu")(x)
x = Dropout(0.2)(x)

# LSTM
# return_sequences=True because TCN needs sequence input
x = LSTM(
    64,
    return_sequences=True,
    name="LSTM_64"
)(x)

x = Dropout(0.2)(x)

# TCN
x = tcn_block(
    x,
    filters=64,
    kernel_size=3,
    dilation_rate=2,
    dropout=0.2
)

# Pooling
x = GlobalAveragePooling1D()(x)

# Dense
x = Dense(
    64,
    activation="relu",
    name="Dense_64"
)(x)

x = Dropout(0.2)(x)

# Output
outputs = Dense(
    6,
    activation="softmax",
    name="Output"
)(x)

triple_model = Model(
    inputs=inputs,
    outputs=outputs,
    name="CNN_LSTM_TCN_Hybrid"
)

# ------------------------------------------------------------
# 2. COMPILE
# ------------------------------------------------------------

triple_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0005
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

triple_model.summary()

# ------------------------------------------------------------
# 3. CALLBACKS
# ------------------------------------------------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=1,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

# ------------------------------------------------------------
# 4. TRAIN
# ------------------------------------------------------------

start_time = time.time()

history_triple = triple_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=15,
    batch_size=256,

    callbacks=[
        early_stop,
        reduce_lr
    ],

    class_weight=None,
    shuffle=True,
    verbose=1
)

training_time = time.time() - start_time

# ------------------------------------------------------------
# 5. TEST
# ------------------------------------------------------------

test_loss, test_accuracy = triple_model.evaluate(
    X_test,
    y_test,
    batch_size=256,
    verbose=1
)

# ------------------------------------------------------------
# 6. PREDICTIONS
# ------------------------------------------------------------

y_prob = triple_model.predict(
    X_test,
    batch_size=256,
    verbose=1
)

y_pred = np.argmax(
    y_prob,
    axis=1
)

# ------------------------------------------------------------
# 7. METRICS
# ------------------------------------------------------------

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

class_names = [
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
]

print("\n" + "=" * 65)
print("CNN + LSTM + TCN HYBRID - FINAL TEST RESULT")
print("=" * 65)

print(f"Test Accuracy : {test_accuracy * 100:.2f}%")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Macro F1      : {macro_f1:.4f}")
print(f"Weighted F1   : {weighted_f1:.4f}")
print(f"Training Time : {training_time / 60:.2f} minutes")

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# 8. KEEP THE NAMES USED BY THE DOWNSTREAM CELLS
# ------------------------------------------------------------
proposed_model = triple_model
y_prob_proposed = y_prob
y_pred_proposed = y_pred
test_accuracy_proposed = test_accuracy
macro_f1_proposed = macro_f1

class_f1 = f1_score(y_test, y_pred, average=None, labels=np.arange(6))
print(f"Unhealthy F1      : {class_f1[3]:.4f}")
print(f"Very Unhealthy F1 : {class_f1[4]:.4f}")
print(f"Hazardous F1      : {class_f1[5]:.4f}")

In [ ]:
# STEP 1 — Inspect IAQI-S composite values for EVT

print("IAQI_Composite exists:", "IAQI_Composite" in data.columns)

if "IAQI_Composite" in data.columns:
    print("\nBasic statistics:")
    print(data["IAQI_Composite"].describe())

    print("\nImportant percentiles:")
    for p in [90, 95, 97.5, 99, 99.5]:
        value = data["IAQI_Composite"].quantile(p / 100)
        print(f"{p}% percentile = {value:.2f}")

In [ ]:
# STEP 2 — Count exceedances for candidate EVT thresholds

thresholds = {
    "95%": data["IAQI_Composite"].quantile(0.95),
    "97.5%": data["IAQI_Composite"].quantile(0.975),
    "99%": data["IAQI_Composite"].quantile(0.99),
    "99.5%": data["IAQI_Composite"].quantile(0.995),
    "Hazardous_400": 400
}

n = len(data)

print("Total observations:", n)
print()

for name, u in thresholds.items():
    count = (data["IAQI_Composite"] > u).sum()
    percent = 100 * count / n

    print(
        f"{name:15s} | Threshold = {u:7.2f} | "
        f"Exceedances = {count:7d} | "
        f"{percent:.4f}%"
    )

In [ ]:
# ============================================================
# EVT STEP 3 — Threshold Diagnostics
# Mean Excess + GPD Shape Stability
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import genpareto

x = data["IAQI_Composite"].dropna().to_numpy()

# Candidate thresholds from 95% to 99.5%
quantiles = np.linspace(0.95, 0.995, 10)
thresholds = np.quantile(x, quantiles)

mean_excess = []
shape_params = []
scale_params = []
n_exceed = []

for u in thresholds:

    excess = x[x > u] - u

    n_exceed.append(len(excess))
    mean_excess.append(np.mean(excess))

    # GPD fit: excesses start from zero
    shape, loc, scale = genpareto.fit(
        excess,
        floc=0
    )

    shape_params.append(shape)
    scale_params.append(scale)

# ------------------------------------------------------------
# PRINT TABLE
# ------------------------------------------------------------

print("=" * 78)
print("EVT THRESHOLD DIAGNOSTICS")
print("=" * 78)

for q, u, n, me, xi in zip(
    quantiles,
    thresholds,
    n_exceed,
    mean_excess,
    shape_params
):
    print(
        f"Q={q:.3f} | "
        f"Threshold={u:7.2f} | "
        f"Exceedances={n:7d} | "
        f"Mean Excess={me:7.2f} | "
        f"Shape ξ={xi:7.4f}"
    )

# ------------------------------------------------------------
# 1. MEAN EXCESS PLOT
# ------------------------------------------------------------

plt.figure(figsize=(8,5))

plt.plot(
    thresholds,
    mean_excess,
    marker="o"
)

plt.xlabel("Threshold (IAQI_Composite)")
plt.ylabel("Mean Excess")
plt.title("Mean Excess Plot")

plt.grid(True)
plt.show()

# ------------------------------------------------------------
# 2. GPD SHAPE STABILITY
# ------------------------------------------------------------

plt.figure(figsize=(8,5))

plt.plot(
    thresholds,
    shape_params,
    marker="o"
)

plt.axhline(
    0,
    linestyle="--"
)

plt.xlabel("Threshold (IAQI_Composite)")
plt.ylabel("GPD Shape Parameter (ξ)")
plt.title("GPD Shape Stability Plot")

plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# EVT STEP 4 — Fit GPD to exceedances above threshold = 152
# ============================================================

import numpy as np
from scipy.stats import genpareto

# Fixed threshold from diagnostics
u = 152.0

# Original IAQI-S values
x = data["IAQI_Composite"].dropna().to_numpy()

# Exceedances above threshold
exceedances = x[x > u] - u

# Fit Generalized Pareto Distribution
shape_xi, loc, scale_beta = genpareto.fit(
    exceedances,
    floc=0
)

# Basic probabilities
p_exceed_u = len(exceedances) / len(x)

print("=" * 70)
print("GPD FIT FOR IAQI_COMPOSITE")
print("=" * 70)

print(f"Threshold u              : {u:.2f}")
print(f"Total observations       : {len(x)}")
print(f"Number of exceedances    : {len(exceedances)}")
print(f"Exceedance probability   : {p_exceed_u:.6f}")

print("\nGPD Parameters:")
print(f"Shape parameter xi       : {shape_xi:.4f}")
print(f"Scale parameter beta     : {scale_beta:.4f}")

print("\nExceedance statistics:")
print(f"Mean excess              : {np.mean(exceedances):.4f}")
print(f"Median excess            : {np.median(exceedances):.4f}")
print(f"Maximum excess           : {np.max(exceedances):.4f}")

# ------------------------------------------------------------
# Estimate probability of exceeding higher IAQI-S levels
# P(X > z) = P(X > u) * P(excess > z-u | X>u)
# ------------------------------------------------------------

levels = [200, 250, 300, 400]

print("\nEstimated exceedance probabilities:")

for z in levels:

    if z <= u:
        continue

    excess_z = z - u

    conditional_survival = genpareto.sf(
        excess_z,
        c=shape_xi,
        loc=0,
        scale=scale_beta
    )

    probability = p_exceed_u * conditional_survival

    print(
        f"P(IAQI > {z}) = "
        f"{probability:.8f} "
        f"({probability*100:.5f}%)"
    )

In [ ]:
# ============================================================
# MARKOV CHAIN - STEP 1
# Transition Matrix for 6 IAQ Classes
# ============================================================

import numpy as np
import pandas as pd

# Class order
classes = [
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
]

# Make a copy and sort correctly
mc = data[
    ["roomcode", "devcode", "segment_id", "Timestamp", "IAQI_Class"]
].copy()

mc = mc.sort_values(
    ["roomcode", "devcode", "segment_id", "Timestamp"]
).reset_index(drop=True)

# Next state only within same room/device/segment
mc["Next_Class"] = (
    mc.groupby(
        ["roomcode", "devcode", "segment_id"]
    )["IAQI_Class"]
    .shift(-1)
)

# Remove final rows of each segment
mc = mc.dropna(subset=["Next_Class"])

# Count transitions
transition_counts = pd.crosstab(
    mc["IAQI_Class"],
    mc["Next_Class"]
)

# Force full 6x6 order
transition_counts = transition_counts.reindex(
    index=classes,
    columns=classes,
    fill_value=0
)

# Convert counts to probabilities
transition_matrix = transition_counts.div(
    transition_counts.sum(axis=1),
    axis=0
)

print("=" * 80)
print("MARKOV CHAIN - TRANSITION COUNTS")
print("=" * 80)
print(transition_counts)

print("\n" + "=" * 80)
print("MARKOV CHAIN - TRANSITION PROBABILITY MATRIX")
print("=" * 80)
print(transition_matrix.round(4))

In [ ]:
# ============================================================
# MARKOV CHAIN - STEP 2
# Expected Residence Time
# ============================================================

import pandas as pd
import numpy as np

sampling_minutes = 3

results = []

for state in classes:

    p_stay = transition_matrix.loc[state, state]

    if p_stay < 1:
        expected_steps = 1 / (1 - p_stay)
        expected_minutes = expected_steps * sampling_minutes
    else:
        expected_steps = np.inf
        expected_minutes = np.inf

    results.append({
        "State": state,
        "P_stay": p_stay,
        "Expected_Steps": expected_steps,
        "Expected_Minutes": expected_minutes
    })

residence_df = pd.DataFrame(results)

print("=" * 75)
print("EXPECTED RESIDENCE TIME")
print("=" * 75)

print(
    residence_df.to_string(
        index=False,
        formatters={
            "P_stay": "{:.4f}".format,
            "Expected_Steps": "{:.2f}".format,
            "Expected_Minutes": "{:.2f}".format
        }
    )
)

In [ ]:
# ============================================================
# MARKOV CHAIN - STEP 3
# Stationary Distribution
# ============================================================

import numpy as np
import pandas as pd

# transition_matrix exists from the previous step

P = transition_matrix.to_numpy()

# Solve pi P = pi
# Equivalent to eigenvector of P.T associated with eigenvalue 1

eigenvalues, eigenvectors = np.linalg.eig(P.T)

# Find eigenvalue closest to 1
idx = np.argmin(np.abs(eigenvalues - 1))

stationary = np.real(eigenvectors[:, idx])

# Normalize
stationary = stationary / stationary.sum()

stationary_df = pd.DataFrame({
    "State": transition_matrix.index,
    "Stationary_Probability": stationary,
    "Expected_Percentage": stationary * 100
})

print("=" * 70)
print("MARKOV CHAIN - STATIONARY DISTRIBUTION")
print("=" * 70)

print(
    stationary_df.to_string(
        index=False,
        formatters={
            "Stationary_Probability": "{:.6f}".format,
            "Expected_Percentage": "{:.3f}%".format
        }
    )
)

print("\nCheck sum =", stationary.sum())

In [ ]:
# ============================================================
# SHAP FEATURE IMPORTANCE
# Proposed Model
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

# ------------------------------------------------------------
# 1. Feature names
# ------------------------------------------------------------

feature_names = [
    "CO2",
    "PM2.5",
    "PM10",
    "T",
    "RH"
]

# ------------------------------------------------------------
# 2. Use a small background + test sample
# ------------------------------------------------------------

N_BACKGROUND = 100
N_EXPLAIN = 200

background = X_train[:N_BACKGROUND]
X_explain = X_test[:N_EXPLAIN]

# ------------------------------------------------------------
# 3. SHAP Explainer
# GradientExplainer works with deep TensorFlow models
# ------------------------------------------------------------

explainer = shap.GradientExplainer(
    proposed_model,
    background
)

shap_values = explainer.shap_values(
    X_explain
)

# ------------------------------------------------------------
# 4. Convert SHAP output to one array
# Expected shape roughly:
# (samples, timesteps, features, classes)
# or list by class depending on SHAP version
# ------------------------------------------------------------

if isinstance(shap_values, list):
    shap_arr = np.stack(shap_values, axis=-1)
else:
    shap_arr = np.array(shap_values)

print("SHAP raw shape:", shap_arr.shape)

# ------------------------------------------------------------
# 5. Global feature importance
# Average absolute SHAP across:
# samples + timesteps + classes
# ------------------------------------------------------------

if shap_arr.ndim == 4:
    feature_importance = np.mean(
        np.abs(shap_arr),
        axis=(0, 1, 3)
    )

elif shap_arr.ndim == 3:
    # fallback if classes already aggregated
    feature_importance = np.mean(
        np.abs(shap_arr),
        axis=(0, 1)
    )

else:
    raise ValueError(
        f"Unexpected SHAP shape: {shap_arr.shape}"
    )

# Normalize
feature_importance = (
    feature_importance /
    feature_importance.sum()
)

# ------------------------------------------------------------
# 6. Table
# ------------------------------------------------------------

shap_df = pd.DataFrame({
    "Feature": feature_names,
    "SHAP_Importance": feature_importance,
    "Percentage": feature_importance * 100
})

shap_df = shap_df.sort_values(
    "SHAP_Importance",
    ascending=False
)

print("\n" + "=" * 60)
print("GLOBAL SHAP FEATURE IMPORTANCE")
print("=" * 60)

print(
    shap_df.to_string(
        index=False,
        formatters={
            "SHAP_Importance": "{:.6f}".format,
            "Percentage": "{:.2f}%".format
        }
    )
)

# ------------------------------------------------------------
# 7. Plot
# ------------------------------------------------------------

plt.figure(figsize=(8,5))

plt.bar(
    shap_df["Feature"],
    shap_df["SHAP_Importance"]
)

plt.xlabel("Feature")
plt.ylabel("Mean |SHAP Value|")
plt.title(
    "Global SHAP Feature Importance "
    "(Proposed Model)"
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.show()

In [ ]:
# ==========================================
# RULE-BASED DECISION LAYER
# ==========================================

import numpy as np
import pandas as pd

predicted_all = np.argmax(y_prob_proposed, axis=1)

class_names = [
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
]

decision_map = {
    0: "Normal",
    1: "Monitor",
    2: "Warning",
    3: "Critical",
    4: "Immediate Action",
    5: "Immediate Action"
}

decision_labels = np.array(
    [decision_map[c] for c in predicted_all]
)

print("Decision distribution:")
print(pd.Series(decision_labels).value_counts())

print("\nExample:")
print("Predicted Class :", class_names[predicted_all[0]])
print("Decision        :", decision_labels[0])

In [ ]:
# ============================================================
# BALANCED DASHBOARD SAMPLE SELECTION
# Includes examples from all IAQ classes
# ============================================================

import numpy as np

feature_names = ["CO2", "PM2.5", "PM10", "T", "RH"]

class_names = np.array([
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
])

# ------------------------------------------------------------
# Model outputs
# ------------------------------------------------------------
confidence_all = np.max(y_prob_proposed, axis=1)
predicted_all = np.argmax(y_prob_proposed, axis=1)

class_levels = np.arange(6)

severity_all = np.sum(
    y_prob_proposed * class_levels,
    axis=1
)

# ------------------------------------------------------------
# Select samples from every predicted class
# ------------------------------------------------------------
samples_per_class = 100

selected_idx = []

for class_id in range(6):
    idx = np.where(predicted_all == class_id)[0]

    if len(idx) > 0:
        take = min(samples_per_class, len(idx))
        selected_idx.extend(idx[:take])

selected_idx = np.array(selected_idx)

print("Selected dashboard samples:", len(selected_idx))

# ------------------------------------------------------------
# Extract selected windows
# ------------------------------------------------------------
X_dash_scaled = X_test[selected_idx]

X_dash_original = np.empty_like(
    X_dash_scaled,
    dtype=np.float32
)

for i in range(len(X_dash_scaled)):
    X_dash_original[i] = scaler.inverse_transform(
        X_dash_scaled[i]
    )

# ------------------------------------------------------------
# Corresponding outputs
# ------------------------------------------------------------
pred_dash = predicted_all[selected_idx]
conf_dash = confidence_all[selected_idx]
severity_dash = severity_all[selected_idx]

action_score_dash = severity_all[selected_idx] / 5 * 100
action_label_dash = decision_labels[selected_idx]

# ------------------------------------------------------------
# Save dashboard dataset
# ------------------------------------------------------------
np.savez(
    "iaq_dashboard_data.npz",

    X=X_dash_original,

    predicted_class=pred_dash,
    confidence=conf_dash,
    severity=severity_dash,

    action_score=action_score_dash,
    action_label=action_label_dash,

    feature_names=np.array(feature_names),
    class_names=class_names,

    original_test_index=selected_idx
)

# ------------------------------------------------------------
# Check distribution
# ------------------------------------------------------------
print("\nDashboard class distribution:")

for class_id, name in enumerate(class_names):
    count = np.sum(pred_dash == class_id)
    print(f"{name}: {count}")

print("\nDashboard Decision action distribution:")

unique_actions, counts = np.unique(
    action_label_dash,
    return_counts=True
)

for a, c in zip(unique_actions, counts):
    print(f"{a}: {c}")

print("\nBALANCED DASHBOARD DATA READY ✅")

In [ ]:
%%writefile app.py

import streamlit as st
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ============================================================
# PAGE SETTINGS
# ============================================================

st.set_page_config(
    page_title="Smart Indoor Air Quality Dashboard",
    page_icon="🌿",
    layout="wide"
)

# ============================================================
# LOAD DATA
# ============================================================

data = np.load(
    "iaq_dashboard_data.npz",
    allow_pickle=True
)

X = data["X"]

predicted_class = data["predicted_class"]
confidence = data["confidence"]
severity = data["severity"]

action_score = data["action_score"]
action_label = data["action_label"]

feature_names = data["feature_names"]
class_names = data["class_names"]

# ============================================================
# HEADER
# ============================================================

st.title("🌿 Smart Indoor Air Quality Monitoring System")

st.caption(
    "Deep Learning (CNN + LSTM + TCN) + Rule-based Decision Support"
)

# ============================================================
# SAMPLE SELECTOR
# ============================================================

sample_id = st.sidebar.slider(
    "Select test sample",
    0,
    len(X) - 1,
    0
)

window = X[sample_id]

# Last reading = current state t-1
current = window[-1]

co2 = current[0]
pm25 = current[1]
pm10 = current[2]
temp = current[3]
rh = current[4]

pred_id = int(predicted_class[sample_id])
pred_name = str(class_names[pred_id])

conf = float(confidence[sample_id])
sev = float(severity[sample_id])

act_score = float(action_score[sample_id])
act_label = str(action_label[sample_id])

# ============================================================
# CURRENT SENSOR VALUES
# ============================================================

st.subheader("Current Indoor Air Quality")

c1, c2, c3, c4, c5 = st.columns(5)

c1.metric(
    "CO₂",
    f"{co2:.1f} ppm"
)

c2.metric(
    "PM2.5",
    f"{pm25:.1f} µg/m³"
)

c3.metric(
    "PM10",
    f"{pm10:.1f} µg/m³"
)

c4.metric(
    "Temperature",
    f"{temp:.1f} °C"
)

c5.metric(
    "Relative Humidity",
    f"{rh:.1f} %"
)

# ============================================================
# MODEL + DECISION OUTPUT
# ============================================================

st.subheader("AI Classification and Decision Support")

d1, d2, d3, d4 = st.columns(4)

d1.metric(
    "Predicted Class",
    pred_name
)

d2.metric(
    "Confidence",
    f"{conf * 100:.2f}%"
)

d3.metric(
    "Severity Score",
    f"{sev:.3f}"
)

d4.metric(
    "Decision Action",
    act_label
)

st.caption(
    f"Severity Index: {act_score:.2f} / 100"
)

# ============================================================
# ALERT MESSAGE
# ============================================================

if act_label == "Normal":

    st.success(
        "✅ Normal — indoor air quality is stable."
    )

elif act_label == "Monitor":

    st.info(
        "ℹ️ Monitor — continue observing indoor air quality."
    )

elif act_label == "Warning":

    st.warning(
        "⚠️ Warning — air quality conditions require attention."
    )

elif act_label == "Critical":

    st.error(
        "🚨 Critical — poor indoor air quality detected."
    )

else:

    st.error(
        "🚨 IMMEDIATE ACTION — severe indoor air quality condition detected."
    )

# ============================================================
# TIME WINDOW
# ============================================================

time_steps = [
    f"t-{len(window)-i}"
    for i in range(len(window))
]

trend = pd.DataFrame(
    window,
    columns=feature_names
)

trend["Time"] = time_steps

# ============================================================
# CO2 TREND
# ============================================================

st.subheader("Temporal Air Quality Trends")

fig_co2 = go.Figure()

fig_co2.add_trace(
    go.Scatter(
        x=trend["Time"],
        y=trend["CO2"],
        mode="lines+markers",
        name="CO₂",
        line=dict(color="#1976D2", width=3)
    )
)

fig_co2.update_layout(
    title="CO₂ Trend",
    xaxis_title="Temporal Window",
    yaxis_title="CO₂ (ppm)",
    height=350
)

st.plotly_chart(
    fig_co2,
    use_container_width=True
)

# ============================================================
# PARTICULATE MATTER
# ============================================================

fig_pm = go.Figure()

fig_pm.add_trace(
    go.Scatter(
        x=trend["Time"],
        y=trend["PM2.5"],
        mode="lines+markers",
        name="PM2.5",
        line=dict(color="#FB8C00", width=3)
    )
)

fig_pm.add_trace(
    go.Scatter(
        x=trend["Time"],
        y=trend["PM10"],
        mode="lines+markers",
        name="PM10",
        line=dict(color="#D32F2F", width=3)
    )
)

fig_pm.update_layout(
    title="Particulate Matter Trend",
    xaxis_title="Temporal Window",
    yaxis_title="µg/m³",
    height=350
)

st.plotly_chart(
    fig_pm,
    use_container_width=True
)

# ============================================================
# TEMPERATURE + HUMIDITY
# ============================================================

fig_env = go.Figure()

fig_env.add_trace(
    go.Scatter(
        x=trend["Time"],
        y=trend["T"],
        mode="lines+markers",
        name="Temperature",
        line=dict(color="#7B1FA2", width=3)
    )
)

fig_env.add_trace(
    go.Scatter(
        x=trend["Time"],
        y=trend["RH"],
        mode="lines+markers",
        name="RH",
        line=dict(color="#388E3C", width=3)
    )
)

fig_env.update_layout(
    title="Temperature and Relative Humidity",
    xaxis_title="Temporal Window",
    yaxis_title="Value",
    height=350
)

st.plotly_chart(
    fig_env,
    use_container_width=True
)

# ============================================================
# SYSTEM INFORMATION
# ============================================================

with st.expander("System Details"):

    st.write("Model: CNN + LSTM + TCN")

    st.write("Window Size: 10 readings")

    st.write("Features: CO₂, PM2.5, PM10, Temperature, Relative Humidity")

    st.write("Decision Layer: Rule-based mapping from the predicted IAQI-S class to an action level")

    st.write("Current Mode: Test-set simulation")

In [ ]:
!python -m pip install -q streamlit plotly

In [ ]:
!python -m streamlit run app.py --server.port 8501 > /tmp/streamlit.log 2>&1 &

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
!nohup ./cloudflared tunnel --url http://localhost:8501 > /tmp/cloudflared.log 2>&1 &

In [ ]:
import time
import re

time.sleep(5)

with open("/tmp/cloudflared.log", "r") as f:
    log = f.read()

match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", log)

if match:
    print("PUBLIC URL:")
    print(match.group(0))
else:
    print(log[-2000:])

In [ ]:
import firebase_admin
from firebase_admin import credentials, db

DB_URL = "YOUR_FIREBASE_DATABASE_URL"
SERVICE_ACCOUNT_FILE = "/content/serviceAccount.json"

if firebase_admin._apps:
    firebase_admin.delete_app(firebase_admin.get_app())

cred = credentials.Certificate(SERVICE_ACCOUNT_FILE)
firebase_admin.initialize_app(cred, {"databaseURL": DB_URL})

def send_to_firebase(data: dict):
    db.reference("current_reading").set(data)
    print("Firebase write OK ✅")

print("FIREBASE READY ✅")

In [ ]:
class_names = ["Good", "Moderate", "Caution", "Unhealthy", "Very Unhealthy", "Hazardous"]

sample_id = int(np.random.randint(0, len(X_test)))
print("Sample index:", sample_id)

current = scaler.inverse_transform(X_test[sample_id][-1].reshape(1, -1))[0]
co2, pm25, pm10, temp, rh = current

pred_id = int(np.argmax(y_prob_proposed[sample_id]))
confidence = float(np.max(y_prob_proposed[sample_id]))
severity = float(np.sum(y_prob_proposed[sample_id] * np.arange(6)))

firebase_data = {
    "CO2": round(float(co2), 2),
    "PM2_5": round(float(pm25), 2),
    "PM10": round(float(pm10), 2),
    "Temperature": round(float(temp), 2),
    "RH": round(float(rh), 2),
    "Predicted_Class": class_names[pred_id],
    "Confidence": round(confidence, 4),
    "Severity_Score": round(severity, 4),
    "Rule_Action": str(action_labels_all[sample_id]),
    "Rule_Action_Score": round(float(action_scores_all[sample_id]), 2),
}

send_to_firebase(firebase_data)
print(firebase_data)

In [ ]:
action_labels_all = decision_labels
action_scores_all = severity_all / 5 * 100
print("Decision source switched to the rule-based layer ✅")
print(np.unique(action_labels_all, return_counts=True))

In [ ]:
idxs = np.where(action_labels_all == "Immediate Action")[0]

if len(idxs) == 0:
    print("No Immediate Action sample found.")
else:
    i = int(idxs[0])
    x = scaler.inverse_transform(X_test[i, -1].reshape(1, -1))[0]
    pred_idx = int(np.argmax(y_prob_proposed[i]))

    payload = {
        "CO2": round(float(x[0]), 2),
        "PM2_5": round(float(x[1]), 2),
        "PM10": round(float(x[2]), 2),
        "Temperature": round(float(x[3]), 2),
        "RH": round(float(x[4]), 2),
        "Predicted_Class": class_names[pred_idx],
        "Confidence": round(float(np.max(y_prob_proposed[i])), 4),
        "Rule_Action": str(action_labels_all[i]),
        "Rule_Action_Score": round(float(action_scores_all[i]), 2),
    }

    send_to_firebase(payload)
    print("Index:", i, "|", payload)

In [ ]:
predicted_all = np.argmax(y_prob_proposed, axis=1)
haz_idx = np.where(predicted_all == 5)[0]

if len(haz_idx) == 0:
    print("No Hazardous prediction found.")
else:
    sample_id = int(haz_idx[0])
    c = scaler.inverse_transform(X_test[sample_id][-1].reshape(1, -1))[0]

    firebase_data = {
        "CO2": round(float(c[0]), 2),
        "PM2_5": round(float(c[1]), 2),
        "PM10": round(float(c[2]), 2),
        "Temperature": round(float(c[3]), 2),
        "RH": round(float(c[4]), 2),
        "Predicted_Class": "Hazardous",
        "Confidence": round(float(np.max(y_prob_proposed[sample_id])), 4),
        "Severity_Score": round(float(np.sum(y_prob_proposed[sample_id] * np.arange(6))), 4),
        "Rule_Action": str(action_labels_all[sample_id]),
        "Rule_Action_Score": round(float(action_scores_all[sample_id]), 2),
    }

    send_to_firebase(firebase_data)
    print("Sample ID:", sample_id, "|", firebase_data)

In [ ]:
import os
import joblib
import numpy as np

SAVE_DIR = "/content/drive/MyDrive/IAQ_Project_Saved"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1) Save trained proposed model
proposed_model.save(f"{SAVE_DIR}/proposed_model.keras")

# 2) Save scaler
joblib.dump(scaler, f"{SAVE_DIR}/scaler.pkl")

# 3) Save important predictions / outputs if they exist
np.save(f"{SAVE_DIR}/y_prob_proposed.npy", y_prob_proposed)
np.save(f"{SAVE_DIR}/y_test.npy", y_test)

print("MODEL + SCALER + PREDICTIONS SAVED ✅")
print("Saved in:", SAVE_DIR)

In [ ]:
# Save Rule-Based Decision outputs
np.save(f"{SAVE_DIR}/action_scores_all.npy", action_scores_all)
np.save(f"{SAVE_DIR}/action_labels_all.npy", action_labels_all)

print("Decision OUTPUTS SAVED ✅")

In [ ]:
np.save(f"{SAVE_DIR}/X_test.npy", X_test)

print("X_test SAVED ✅")
print("Shape:", X_test.shape)

In [ ]:
DEVICE_TOKEN = "YOUR_DEVICE_TOKEN"

In [ ]:
from firebase_admin import messaging

def send_iaq_notification(rule_action, predicted_class):
    if rule_action == "Warning":
        title = "IAQ Warning"
        body = "Air quality is deteriorating. Activate ventilation."

    elif rule_action == "Critical":
        title = "Critical IAQ Alert"
        body = "Poor air quality detected. Ventilation and alarm are required."

    elif rule_action == "Immediate Action":
        title = "Urgent IAQ Alert"
        body = "Severe indoor air quality detected. Immediate action is required."

    else:
        print("No FCM notification needed.")
        return

    message = messaging.Message(
        notification=messaging.Notification(
            title=title,
            body=body
        ),
        token=DEVICE_TOKEN
    )

    response = messaging.send(message)
    print("FCM SENT ✅")
    print("Class:", predicted_class)
    print("Action:", rule_action)
    print("Message ID:", response)

In [ ]:
class_names = [
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
]

pred_idx = int(np.argmax(y_prob_proposed[sample_id]))
pred_class = class_names[pred_idx]

send_iaq_notification(
    rule_action=str(action_labels_all[sample_id]),
    predicted_class=pred_class
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

class_names = [
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
]

y_pred = np.argmax(y_prob_proposed, axis=1)

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(6)
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True {x}" for x in class_names],
    columns=[f"Pred {x}" for x in class_names]
)

display(cm_df)

In [ ]:
# ============================================================
# What share of the rows is interpolated rather than actually measured?
# Second file (proposed model) — placed as the last cell
# Requirements: run the whole notebook first (no GPU needed, no retraining)
# Variables required in memory: work, df_resampled, train_part, val_part,
#   test_part, y_test, y_pred_proposed, SENSOR_COLS
# ============================================================
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

WINDOW = 10
KEYS = ["roomcode", "devcode", "Timestamp"]
SENSOR_COLS = ["CO2", "PM2.5", "PM10", "T", "RH"]

# ------------------------------------------------------------------
# 1) For each (room, device, 3-minute bin): was there a real measurement for each sensor?
# ------------------------------------------------------------------
w = work[KEYS + SENSOR_COLS].copy()
w["Timestamp"] = w["Timestamp"].dt.floor("3min")

flags = df_resampled[KEYS].drop_duplicates().reset_index(drop=True)
for c in SENSOR_COLS:
    obs = w.loc[w[c].notna(), KEYS].drop_duplicates()
    obs["real_" + c] = True
    flags = flags.merge(obs, on=KEYS, how="left")
    flags["real_" + c] = flags["real_" + c].fillna(False).astype(bool)

REAL_COLS = ["real_" + c for c in SENSOR_COLS]
flags["all_real"] = flags[REAL_COLS].all(axis=1)

# ------------------------------------------------------------------
# 2) Shares on the full data and on each split (Train / Val / Test)
# ------------------------------------------------------------------
def attach(part):
    out = part.merge(flags[KEYS + REAL_COLS + ["all_real"]], on=KEYS, how="left")
    for c in REAL_COLS + ["all_real"]:
        out[c] = out[c].fillna(False).astype(bool)
    assert len(out) == len(part), "Merge changed row count"
    return out

train_f, val_f, test_f = attach(train_part), attach(val_part), attach(test_part)

def share(df, name):
    n = len(df)
    row = {"set": name, "rows": n,
           "measured_all_5 %": 100 * df["all_real"].mean()}
    for c in SENSOR_COLS:
        row[f"{c} measured %"] = 100 * df["real_" + c].mean()
    return row

table = pd.DataFrame([
    share(flags.merge(df_resampled[KEYS], on=KEYS), "All resampled rows"),
    share(train_f, "Train"),
    share(val_f, "Validation"),
    share(test_f, "Test"),
]).round(2)
table["interpolated %"] = (100 - table["measured_all_5 %"]).round(2)

print("=" * 78)
print("Share of rows actually measured vs interpolated")
print("=" * 78)
print(table.to_string(index=False))

# ------------------------------------------------------------------
# 3) Sensitivity test: performance of the proposed model on Test windows
#    whose ten steps are all actually measured vs all windows
#    (exactly the same order as the make_windows windows)
# ------------------------------------------------------------------
def window_flags(df, flag_col, window=WINDOW):
    out = []
    for _, g in df.groupby(["roomcode", "devcode", "segment_id"], sort=False):
        if len(g) < window:
            continue
        g = g.sort_values("Timestamp")
        f = g[flag_col].to_numpy(dtype=bool)
        sw = np.lib.stride_tricks.sliding_window_view(f, window)
        out.append(sw.all(axis=1))
    return np.concatenate(out)

real_win = window_flags(test_f, "all_real")
assert len(real_win) == len(y_test), (
    f"window count mismatch: {len(real_win)} vs {len(y_test)}"
)

def metrics(mask, name):
    yt, yp = y_test[mask], np.asarray(y_pred_proposed)[mask]
    f1c = f1_score(yt, yp, average=None, labels=np.arange(6), zero_division=0)
    return {"subset": name, "windows": int(mask.sum()),
            "Hazardous windows": int((yt == 5).sum()),
            "accuracy %": round(100 * accuracy_score(yt, yp), 2),
            "macro-F1": round(f1_score(yt, yp, average="macro", zero_division=0), 4),
            "Hazardous F1": round(f1c[5], 4)}

res = pd.DataFrame([
    metrics(np.ones(len(y_test), dtype=bool), "All test windows"),
    metrics(real_win, "Only windows with 10 measured steps"),
    metrics(~real_win, "Windows containing interpolated steps"),
])
print("\n" + "=" * 78)
print("Performance of the proposed model by window type (Test)")
print("=" * 78)
print(res.to_string(index=False))
print(f"\nShare of test windows fully measured: {100 * real_win.mean():.2f}%")

In [ ]:
import numpy as np
names = ["Good","Moderate","Caution","Unhealthy","Very Unhealthy","Hazardous"]
print(f"{'Class':<16}{'Train':>10}{'Val':>10}{'Test':>10}")
for i, n in enumerate(names):
    print(f"{n:<16}{(y_train==i).sum():>10}{(y_val==i).sum():>10}{(y_test==i).sum():>10}")
print(f"{'Total':<16}{len(y_train):>10}{len(y_val):>10}{len(y_test):>10}")

In [ ]:
# ============================================================
# BAQUBAH PAPER MODEL
# 1D-CNN + BiLSTM
# Tested on OUR dataset and OUR experimental conditions
# ============================================================

import time
import random
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    BatchNormalization,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.metrics import (
    classification_report,
    f1_score
)

# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# ============================================================
# 2. INPUT
# ============================================================

inputs = Input(
    shape=X_train.shape[1:],
    name="Input"
)

# ============================================================
# 3. 1D-CNN
# Local feature extraction
# ============================================================

x = Conv1D(
    filters=128,
    kernel_size=3,
    padding="same",
    activation="relu",
    name="Conv1D_128"
)(inputs)

x = BatchNormalization(
    name="CNN_BatchNorm"
)(x)

x = MaxPooling1D(
    pool_size=2,
    name="MaxPooling"
)(x)

# ============================================================
# 4. FIRST BiLSTM
# Bidirectional temporal modelling
# ============================================================

x = Bidirectional(
    LSTM(
        128,
        return_sequences=True
    ),
    name="BiLSTM_128"
)(x)

x = Dropout(
    0.20,
    name="BiLSTM1_Dropout"
)(x)

# ============================================================
# 5. SECOND BiLSTM
# ============================================================

x = Bidirectional(
    LSTM(
        64,
        return_sequences=False
    ),
    name="BiLSTM_64"
)(x)

x = BatchNormalization(
    name="BiLSTM_BatchNorm"
)(x)

x = Dropout(
    0.20,
    name="BiLSTM2_Dropout"
)(x)

# ============================================================
# 6. FULLY CONNECTED LAYER
# ============================================================

x = Dense(
    64,
    activation="relu",
    name="Dense_64"
)(x)

x = Dropout(
    0.20,
    name="Dense_Dropout"
)(x)

# ============================================================
# 7. OUTPUT
# OUR DATASET = 6 IAQ CLASSES
# ============================================================

outputs = Dense(
    6,
    activation="softmax",
    name="Output"
)(x)

baqubah_model = Model(
    inputs=inputs,
    outputs=outputs,
    name="Baqubah_1DCNN_BiLSTM"
)

# ============================================================
# 8. COMPILE
# Same learning rate as OUR comparison
# ============================================================

baqubah_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0005
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

baqubah_model.summary()

# ============================================================
# 9. CALLBACKS
# Same policy as our models
# ============================================================

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=1,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

# ============================================================
# 10. TRAIN
# ============================================================

start_time = time.time()

history_baqubah = baqubah_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=15,
    batch_size=256,

    class_weight=None,

    callbacks=[
        early_stop,
        reduce_lr
    ],

    shuffle=True,
    verbose=1
)
training_time_baqubah = time.time() - start_time

# ============================================================
# 11. TEST
# ============================================================

test_loss_baqubah, test_acc_baqubah = \
    baqubah_model.evaluate(
        X_test,
        y_test,
        batch_size=256,
        verbose=1
    )

# ============================================================
# 12. PREDICTIONS
# ============================================================

y_prob_baqubah = baqubah_model.predict(
    X_test,
    batch_size=256,
    verbose=1
)

y_pred_baqubah = np.argmax(
    y_prob_baqubah,
    axis=1
)

# ============================================================
# 13. METRICS
# ============================================================

macro_f1_baqubah = f1_score(
    y_test,
    y_pred_baqubah,
    average="macro"
)

weighted_f1_baqubah = f1_score(
    y_test,
    y_pred_baqubah,
    average="weighted"
)

class_f1_baqubah = f1_score(
    y_test,
    y_pred_baqubah,
    labels=np.arange(6),
    average=None
)

mean_rare_f1_baqubah = np.mean(
    class_f1_baqubah[3:6]
)

class_names = [
    "Good",
    "Moderate",
    "Caution",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous"
]

# ============================================================
# 14. FINAL RESULTS
# ============================================================

print("\n" + "=" * 78)
print("BAQUBAH PAPER MODEL: 1D-CNN + BiLSTM")
print("Tested on OUR dataset")
print("=" * 78)

print(f"Test Accuracy       : {test_acc_baqubah*100:.2f}%")
print(f"Test Loss           : {test_loss_baqubah:.4f}")
print(f"Macro F1            : {macro_f1_baqubah:.4f}")
print(f"Weighted F1         : {weighted_f1_baqubah:.4f}")
print(f"Mean Rare F1        : {mean_rare_f1_baqubah:.4f}")

print(f"Unhealthy F1        : {class_f1_baqubah[3]:.4f}")
print(f"Very Unhealthy F1   : {class_f1_baqubah[4]:.4f}")
print(f"Hazardous F1        : {class_f1_baqubah[5]:.4f}")

print(
    f"Training Time       : "
    f"{training_time_baqubah/60:.2f} minutes"
)

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred_baqubah,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

In [ ]:
import numpy as np
import joblib
import tensorflow as tf

SAVE_DIR = "/content/drive/MyDrive/IAQ_Project_Saved"

# Model
proposed_model = tf.keras.models.load_model(
    f"{SAVE_DIR}/proposed_model.keras"
)

# Scaler
scaler = joblib.load(
    f"{SAVE_DIR}/scaler.pkl"
)

# Test data + labels
X_test = np.load(
    f"{SAVE_DIR}/X_test.npy"
)

y_test = np.load(
    f"{SAVE_DIR}/y_test.npy"
)

# Model predictions
y_prob_proposed = np.load(
    f"{SAVE_DIR}/y_prob_proposed.npy"
)

# Rule-based outputs
action_scores_all = np.load(
    f"{SAVE_DIR}/action_scores_all.npy"
)

action_labels_all = np.load(
    f"{SAVE_DIR}/action_labels_all.npy",
    allow_pickle=True
)

print("EVERYTHING LOADED ✅")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)
print("y_prob_proposed:", y_prob_proposed.shape)
print("action_scores_all:", action_scores_all.shape)
print("action_labels_all:", action_labels_all.shape)

In [ ]:
db.reference("alerts").delete()

hist = db.reference("history").get(shallow=True)
keys = sorted(hist.keys())
old_keys = keys[:-100]
db.reference("history").update({k: None for k in old_keys})

print("alerts:", len(db.reference("alerts").get(shallow=True) or {}))
print("history:", len(db.reference("history").get(shallow=True) or {}))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

names = ["Good", "Moderate", "Caution", "Unhealthy", "Very Unhealthy", "Hazardous"]

# Last reading in each window, in real units
x_last = scaler.inverse_transform(X_test[:, -1, :])
co2, pm25, pm10 = x_last[:, 0], x_last[:, 1], x_last[:, 2]

# US EPA AQI breakpoints (PM2.5 as revised in 2024, and PM10)
pm25_bins = [9.0, 35.4, 55.4, 125.4, 225.4]
pm10_bins = [54, 154, 254, 354, 424]

epa_cls = np.maximum(
    np.digitize(pm25, pm25_bins, right=True),
    np.digitize(pm10, pm10_bins, right=True),
)
y = y_test.astype(int)

def report(mask, title):
    a, b = y[mask], epa_cls[mask]
    print(f"\n{title}  (n={mask.sum():,})")
    print("Exact agreement      :", round((a == b).mean() * 100, 2), "%")
    print("Within 1 class       :", round((np.abs(a - b) <= 1).mean() * 100, 2), "%")
    print("Cohen's kappa        :", round(cohen_kappa_score(a, b), 4))
    print("Weighted kappa (quad):", round(cohen_kappa_score(a, b, weights="quadratic"), 4))

report(np.ones(len(y), bool), "All test windows")
report(co2 < 1000, "Windows with CO2 < 1000 ppm (PM drives the label)")

cm = confusion_matrix(y, epa_cls, labels=range(6))
print("\nRows = IAQI-S label, Columns = EPA class")
print(pd.DataFrame(cm, index=names, columns=names))

In [ ]:
import tensorflow as tf, os

path = "/content/drive/MyDrive/IAQ_Project_Saved/proposed_model.keras"
m = tf.keras.models.load_model(path, compile=False, safe_mode=False)
print("params:", m.count_params())

conv = tf.lite.TFLiteConverter.from_keras_model(m)
try:
    tfl = conv.convert()
    print("converted with built-in ops only ✅")
except Exception as e:
    print("built-in failed:", str(e)[:300])
    conv = tf.lite.TFLiteConverter.from_keras_model(m)
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS,
                                      tf.lite.OpsSet.SELECT_TF_OPS]
    conv._experimental_lower_tensor_list_ops = False
    tfl = conv.convert()
    print("converted but needs SELECT_TF_OPS ⚠️")

out = "/content/drive/MyDrive/IAQ_Project_Saved/proposed_float.tflite"
open(out, "wb").write(tfl)
print("size KB:", round(len(tfl)/1024, 1))

In [ ]:
import tensorflow as tf

path = "/content/drive/MyDrive/IAQ_Project_Saved/proposed_model.keras"
m = tf.keras.models.load_model(path, compile=False, safe_mode=False)

def swap(layer):
    cfg = layer.get_config()
    if isinstance(layer, tf.keras.layers.LSTM):
        cfg["unroll"] = True
    return layer.__class__.from_config(cfg)

m2 = tf.keras.models.clone_model(m, clone_function=swap)
m2.set_weights(m.get_weights())
print("params:", m2.count_params())

conv = tf.lite.TFLiteConverter.from_keras_model(m2)
tfl = conv.convert()
print("converted with built-in ops only ✅")

out = "/content/drive/MyDrive/IAQ_Project_Saved/proposed_float_unrolled.tflite"
open(out, "wb").write(tfl)
print("size KB:", round(len(tfl)/1024, 1))

In [ ]:
import numpy as np, tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score

tfl_path = "/content/drive/MyDrive/IAQ_Project_Saved/proposed_float_unrolled.tflite"
interp = tf.lite.Interpreter(model_path=tfl_path)
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]

BS = 4096
preds = []
for i in range(0, len(X_test), BS):
    xb = X_test[i:i+BS].astype(np.float32)
    interp.resize_tensor_input(inp["index"], [len(xb), 10, 5])
    interp.allocate_tensors()
    interp.set_tensor(inp["index"], xb)
    interp.invoke()
    preds.append(interp.get_tensor(out["index"]).copy())
p_tfl = np.concatenate(preds)

y_tfl = p_tfl.argmax(1)
y_orig = y_prob_proposed.argmax(1)

print("TFLite  acc:", round(accuracy_score(y_test, y_tfl)*100, 2),
      "| macro-F1:", round(f1_score(y_test, y_tfl, average="macro"), 4))
print("Original acc:", round(accuracy_score(y_test, y_orig)*100, 2),
      "| macro-F1:", round(f1_score(y_test, y_orig, average="macro"), 4))
print("agreement:", round((y_tfl == y_orig).mean()*100, 3), "%")
print("max prob diff:", float(np.abs(p_tfl - y_prob_proposed).max()))

In [ ]:
import numpy as np, tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, recall_score

path = "/content/drive/MyDrive/IAQ_Project_Saved/proposed_model.keras"
m = tf.keras.models.load_model(path, compile=False, safe_mode=False)

def swap(layer):
    cfg = layer.get_config()
    if isinstance(layer, tf.keras.layers.LSTM):
        cfg["unroll"] = True
    return layer.__class__.from_config(cfg)

m2 = tf.keras.models.clone_model(m, clone_function=swap)
m2.set_weights(m.get_weights())

# Calibration sample: up to 400 windows per class (so the rare classes are included)
rng = np.random.default_rng(0)
idx = []
for c in np.unique(y_test):
    ci = np.where(y_test == c)[0]
    idx += list(rng.choice(ci, min(400, len(ci)), replace=False))
calib = X_test[np.array(idx)].astype(np.float32)
print("calibration windows:", len(calib))

def rep():
    for i in range(len(calib)):
        yield [calib[i:i+1]]

conv = tf.lite.TFLiteConverter.from_keras_model(m2)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
tfl8 = conv.convert()

out_path = "/content/drive/MyDrive/IAQ_Project_Saved/proposed_int8.tflite"
open(out_path, "wb").write(tfl8)
print("int8 size KB:", round(len(tfl8)/1024, 1))

# Evaluation
interp = tf.lite.Interpreter(model_content=tfl8)
inp = interp.get_input_details()[0]; out = interp.get_output_details()[0]
si, zi = inp["quantization"]; so, zo = out["quantization"]

preds = []
BS = 4096
for i in range(0, len(X_test), BS):
    xb = X_test[i:i+BS].astype(np.float32)
    xq = np.clip(np.round(xb/si + zi), -128, 127).astype(np.int8)
    interp.resize_tensor_input(inp["index"], [len(xb), 10, 5])
    interp.allocate_tensors()
    interp.set_tensor(inp["index"], xq)
    interp.invoke()
    q = interp.get_tensor(out["index"]).astype(np.float32)
    preds.append((q - zo) * so)
p8 = np.concatenate(preds)

y8 = p8.argmax(1)
y_orig = y_prob_proposed.argmax(1)
print("INT8     acc:", round(accuracy_score(y_test, y8)*100, 2),
      "| macro-F1:", round(f1_score(y_test, y8, average="macro"), 4))
print("Original acc:", round(accuracy_score(y_test, y_orig)*100, 2),
      "| macro-F1:", round(f1_score(y_test, y_orig, average="macro"), 4))
print("agreement:", round((y8 == y_orig).mean()*100, 2), "%")
print("recall per class  orig :", np.round(recall_score(y_test, y_orig, average=None), 3))
print("recall per class  int8 :", np.round(recall_score(y_test, y8, average=None), 3))

In [ ]:
import numpy as np, tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, recall_score

# Look for the training (or validation) data in memory
if "X_train" in globals() and "y_train" in globals():
    Xc, yc, src = X_train, y_train, "train"
elif "X_val" in globals() and "y_val" in globals():
    Xc, yc, src = X_val, y_val, "val"
else:
    Xc = None
    print("❌ X_train and X_val were not found in memory. Run the cells that load the data first.")

if Xc is not None:
    print("calibration source:", src, Xc.shape)

    path = "/content/drive/MyDrive/IAQ_Project_Saved/proposed_model.keras"
    m = tf.keras.models.load_model(path, compile=False, safe_mode=False)

    def swap(layer):
        cfg = layer.get_config()
        if isinstance(layer, tf.keras.layers.LSTM):
            cfg["unroll"] = True
        return layer.__class__.from_config(cfg)

    m2 = tf.keras.models.clone_model(m, clone_function=swap)
    m2.set_weights(m.get_weights())

    rng = np.random.default_rng(0)
    idx = []
    for c in np.unique(yc):
        ci = np.where(yc == c)[0]
        idx += list(rng.choice(ci, min(400, len(ci)), replace=False))
    calib = Xc[np.array(idx)].astype(np.float32)
    print("calibration windows:", len(calib))

    def rep():
        for i in range(len(calib)):
            yield [calib[i:i+1]]

    conv = tf.lite.TFLiteConverter.from_keras_model(m2)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = rep
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8
    conv.inference_output_type = tf.int8
    tfl8 = conv.convert()

    open("/content/drive/MyDrive/IAQ_Project_Saved/proposed_int8_clean.tflite", "wb").write(tfl8)
    print("int8 size KB:", round(len(tfl8)/1024, 1))

    interp = tf.lite.Interpreter(model_content=tfl8)
    inp = interp.get_input_details()[0]; out = interp.get_output_details()[0]
    si, zi = inp["quantization"]; so, zo = out["quantization"]

    preds = []
    for i in range(0, len(X_test), 4096):
        xb = X_test[i:i+4096].astype(np.float32)
        xq = np.clip(np.round(xb/si + zi), -128, 127).astype(np.int8)
        interp.resize_tensor_input(inp["index"], [len(xb), 10, 5])
        interp.allocate_tensors()
        interp.set_tensor(inp["index"], xq)
        interp.invoke()
        preds.append((interp.get_tensor(out["index"]).astype(np.float32) - zo) * so)
    y8 = np.concatenate(preds).argmax(1)
    y_orig = y_prob_proposed.argmax(1)

    print("INT8     acc:", round(accuracy_score(y_test, y8)*100, 2),
          "| macro-F1:", round(f1_score(y_test, y8, average="macro"), 4))
    print("Original acc:", round(accuracy_score(y_test, y_orig)*100, 2),
          "| macro-F1:", round(f1_score(y_test, y_orig, average="macro"), 4))
    print("agreement:", round((y8 == y_orig).mean()*100, 2), "%")
    print("recall orig :", np.round(recall_score(y_test, y_orig, average=None), 3))
    print("recall int8 :", np.round(recall_score(y_test, y8, average=None), 3))

In [ ]:
import os
d = "/content/drive/MyDrive/IAQ_Project_Saved"
for f in sorted(os.listdir(d)):
    p = os.path.join(d, f)
    print(f, "|", round(os.path.getsize(p)/1024/1024, 1), "MB")

In [ ]:
import numpy as np, tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, recall_score

d = "/content/drive/MyDrive/IAQ_Project_Saved"
Xc = np.load(f"{d}/X_ext_bd.npy")
yc = np.load(f"{d}/y_ext_bd.npy")
print("calibration source: Bangladesh external", Xc.shape)

m = tf.keras.models.load_model(f"{d}/proposed_model.keras", compile=False, safe_mode=False)

def swap(layer):
    cfg = layer.get_config()
    if isinstance(layer, tf.keras.layers.LSTM):
        cfg["unroll"] = True
    return layer.__class__.from_config(cfg)

m2 = tf.keras.models.clone_model(m, clone_function=swap)
m2.set_weights(m.get_weights())

rng = np.random.default_rng(0)
idx = []
for c in np.unique(yc):
    ci = np.where(yc == c)[0]
    idx += list(rng.choice(ci, min(400, len(ci)), replace=False))
calib = Xc[np.array(idx)].astype(np.float32)
print("calibration windows:", len(calib))

def rep():
    for i in range(len(calib)):
        yield [calib[i:i+1]]

conv = tf.lite.TFLiteConverter.from_keras_model(m2)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
tfl8 = conv.convert()

open(f"{d}/proposed_int8_extcal.tflite", "wb").write(tfl8)
print("int8 size KB:", round(len(tfl8)/1024, 1))

interp = tf.lite.Interpreter(model_content=tfl8)
inp = interp.get_input_details()[0]; out = interp.get_output_details()[0]
si, zi = inp["quantization"]; so, zo = out["quantization"]

preds = []
for i in range(0, len(X_test), 4096):
    xb = X_test[i:i+4096].astype(np.float32)
    xq = np.clip(np.round(xb/si + zi), -128, 127).astype(np.int8)
    interp.resize_tensor_input(inp["index"], [len(xb), 10, 5])
    interp.allocate_tensors()
    interp.set_tensor(inp["index"], xq)
    interp.invoke()
    preds.append((interp.get_tensor(out["index"]).astype(np.float32) - zo) * so)
y8 = np.concatenate(preds).argmax(1)
y_orig = y_prob_proposed.argmax(1)

print("INT8     acc:", round(accuracy_score(y_test, y8)*100, 2),
      "| macro-F1:", round(f1_score(y_test, y8, average="macro"), 4))
print("Original acc:", round(accuracy_score(y_test, y_orig)*100, 2),
      "| macro-F1:", round(f1_score(y_test, y_orig, average="macro"), 4))
print("agreement:", round((y8 == y_orig).mean()*100, 2), "%")
print("recall orig :", np.round(recall_score(y_test, y_orig, average=None), 3))
print("recall int8 :", np.round(recall_score(y_test, y8, average=None), 3))

In [ ]:
# Send one Caution-predicted test window to Firebase using the rule-based fields
import numpy as np, joblib
import firebase_admin
from firebase_admin import credentials, db

D = "/content/drive/MyDrive/IAQ_Project_Saved"
DB_URL = "YOUR_FIREBASE_DATABASE_URL"

# Firebase connection
if firebase_admin._apps:
    firebase_admin.delete_app(firebase_admin.get_app())
firebase_admin.initialize_app(credentials.Certificate("/content/serviceAccount.json"), {"databaseURL": DB_URL})

# Saved test windows, model probabilities and scaler
X_test = np.load(f"{D}/X_test.npy")
y_prob = np.load(f"{D}/y_prob_proposed.npy")
scaler = joblib.load(f"{D}/scaler.pkl")

# Rule-based mapping from predicted class to action level
decision_map = {0: "Normal", 1: "Monitor", 2: "Warning", 3: "Critical", 4: "Immediate Action", 5: "Immediate Action"}
pred = y_prob.argmax(1)
severity = (y_prob * np.arange(6)).sum(1)
score = severity / 5 * 100

# Pick a Caution-predicted window
idx = np.where(pred == 2)[0]
i = int(idx[len(idx) // 2])
c = scaler.inverse_transform(X_test[i][-1].reshape(1, -1))[0]

record = {
    "CO2": round(float(c[0]), 2), "PM2_5": round(float(c[1]), 2), "PM10": round(float(c[2]), 2),
    "Temperature": round(float(c[3]), 2), "RH": round(float(c[4]), 2),
    "Predicted_Class": "Caution",
    "Confidence": round(float(y_prob[i].max()), 4),
    "Rule_Action": decision_map[int(pred[i])],
    "Rule_Action_Score": round(float(score[i]), 2),
}
db.reference("current_reading").set(record)
print("Firebase write OK ✅")
print(record)